In [14]:
import os
import requests
from PIL import Image
from glob import glob
import json



# === 사용자 설정 ===
prediction_key = "5sloqMYWiZSpfCd7HZgQ9ZpfuQAXnRWwjSw648WjXGr5Fy5f7imcJQQJ99BFACYeBjFXJ3w3AAAIACOGBDCR"
prediction_url = "https://7aiteam05ai012-prediction.cognitiveservices.azure.com/customvision/v3.0/Prediction/f360eeb2-f8df-441e-8bed-f27d3ef1797a/detect/iterations/Iteration2/image"
input_folder = "C:/Users/USER/Desktop/code_project1/project/medium_heavy_160"
output_folder = "C:/Users/USER/Desktop/code_project1/project/crop_custom_medium_heavy"
fail_folder = os.path.join(output_folder, "fail")
confidence_threshold = 0.5

# === 폴더 생성 ===
os.makedirs(output_folder, exist_ok=True)
os.makedirs(fail_folder, exist_ok=True)

# === 이미지 경로 불러오기 ===
image_paths = glob(os.path.join(input_folder, "*.jpg"))

# === 예측 + 크롭 ===
for image_path in image_paths:
    try:
        # 이미지 열어서 크기 체크
        img = Image.open(image_path)
        width, height = img.size

        if width < 100 or height < 100:
            print(f"⚠️ 너무 작은 이미지 건너뜀: {os.path.basename(image_path)}")
            img.save(os.path.join(fail_folder, os.path.basename(image_path)))
            continue

        # 예측 요청
        with open(image_path, "rb") as img_file:
            headers = {
                "Prediction-Key": prediction_key,
                "Content-Type": "application/octet-stream"
            }
            response = requests.post(prediction_url, headers=headers, data=img_file.read())

        try:
            response_json = response.json()
        except Exception:
            print(f"❌ JSON 파싱 실패: {os.path.basename(image_path)}")
            with open(os.path.join(fail_folder, os.path.basename(image_path)), "wb") as f:
                f.write(open(image_path, "rb").read())
            continue

        # 예측 결과 확인
        if "predictions" not in response_json:
            print(f"❌ 예측 실패: {os.path.basename(image_path)}")
            print("📋 응답 내용:", response_json)

            # 실패 이미지 저장
            img.save(os.path.join(fail_folder, os.path.basename(image_path)))

            # 응답 JSON도 같이 저장
            json_path = os.path.join(fail_folder, os.path.splitext(os.path.basename(image_path))[0] + ".json")
            with open(json_path, "w", encoding="utf-8") as jf:
                json.dump(response_json, jf, ensure_ascii=False, indent=2)

            continue

        predictions = response_json["predictions"]
        cropped_count = 0

        for i, pred in enumerate(predictions):
            if pred["probability"] < confidence_threshold:
                continue

            box = pred["boundingBox"]
            left = int(box["left"] * width)
            top = int(box["top"] * height)
            right = int((box["left"] + box["width"]) * width)
            bottom = int((box["top"] + box["height"]) * height)

            cropped = img.crop((left, top, right, bottom))
            filename = os.path.splitext(os.path.basename(image_path))[0]
            cropped.save(os.path.join(output_folder, f"{filename}_crop{i}.jpg"))
            cropped_count += 1

        print(f"✅ {os.path.basename(image_path)} → {cropped_count}개 크롭 완료")

    except Exception as e:
        print(f"⚠️ {os.path.basename(image_path)} 처리 중 오류 발생: {e}")


✅ hold_101.jpg → 1개 크롭 완료
✅ hold_102.jpg → 1개 크롭 완료
✅ hold_103.jpg → 1개 크롭 완료
✅ hold_104.jpg → 2개 크롭 완료
✅ hold_105.jpg → 1개 크롭 완료
✅ hold_106.jpg → 1개 크롭 완료
✅ hold_107.jpg → 1개 크롭 완료
✅ hold_108.jpg → 1개 크롭 완료
✅ hold_109.jpg → 2개 크롭 완료
✅ hold_11.jpg → 1개 크롭 완료
✅ hold_110.jpg → 1개 크롭 완료
✅ hold_111.jpg → 1개 크롭 완료
✅ hold_112.jpg → 1개 크롭 완료
✅ hold_113.jpg → 1개 크롭 완료
✅ hold_114.jpg → 1개 크롭 완료
✅ hold_115.jpg → 2개 크롭 완료
✅ hold_116.jpg → 1개 크롭 완료
✅ hold_117.jpg → 1개 크롭 완료
✅ hold_118.jpg → 1개 크롭 완료
✅ hold_119.jpg → 1개 크롭 완료
✅ hold_12.jpg → 1개 크롭 완료
✅ hold_120.jpg → 1개 크롭 완료
✅ hold_121.jpg → 1개 크롭 완료
✅ hold_122.jpg → 1개 크롭 완료
✅ hold_123.jpg → 1개 크롭 완료
✅ hold_124.jpg → 2개 크롭 완료
✅ hold_125.jpg → 1개 크롭 완료
✅ hold_126.jpg → 1개 크롭 완료
✅ hold_127.jpg → 1개 크롭 완료
✅ hold_128.jpg → 1개 크롭 완료
✅ hold_129.jpg → 2개 크롭 완료
✅ hold_13.jpg → 2개 크롭 완료
✅ hold_130.jpg → 2개 크롭 완료
✅ hold_131.jpg → 1개 크롭 완료
✅ hold_132.jpg → 1개 크롭 완료
✅ hold_133.jpg → 1개 크롭 완료
✅ hold_14.jpg → 3개 크롭 완료
✅ hold_15.jpg → 1개 크롭 완료
✅ hold_16.jpg → 1